# Week 22, Day 1: One-dimensional flows: fixed points and their stability

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Find and classify the fixed points

Write `fixed_points(w, I)`: scan r from −0.1 to 1.1 on a fine grid, find every sign change of f, refine each root by bisection, and return a list of (r*, stable) pairs, where stable is True when the slope of f at r* is negative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tau = 10.0
F = lambda u: 1 / (1 + np.exp(-(u - 4)))
def f(r, w, I): return -r + F(w * r + I)

def fixed_points(w, I):
    grid = np.linspace(-0.1, 1.1, 2001)
    vals = f(grid, w, I)
    out = []
    for k in np.where(np.sign(vals[:-1]) != np.sign(vals[1:]))[0]:
        a, b = grid[k], grid[k + 1]
        for _ in range(60):                          # bisection
            m = (a + b) / 2
            if ...: b = m
            else: a = m
        r = (a + b) / 2
        slope = (f(r + 1e-6, w, I) - f(r - 1e-6, w, I)) / 2e-6
        out.append((r, ...))
    return out

for w in (2.0, 8.0):
    print(f"w = {w}:", [(round(float(r), 4), 'stable' if s else 'unstable') for r, s in fixed_points(w, 0.0)])
rr = np.linspace(-0.1, 1.1, 300)
for w in (2.0, 8.0): plt.plot(rr, f(rr, w, 0.0), label=f"w = {w}")
plt.axhline(0, c="k", lw=0.8); plt.xlabel("r"); plt.ylabel("tau dr/dt"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_three():
    "with w = 8 and I = 0 there are three fixed points, stable, unstable, stable"
    fps = fixed_points(8.0, 0.0)
    assert len(fps) == 3 and [s for _, s in fps] == [True, False, True]

def test_values():
    "each fixed point really satisfies f(r) = 0 to 1e-8"
    assert all(abs(f(r, 8.0, 0.0)) < 1e-8 for r, _ in fixed_points(8.0, 0.0))

def test_weak():
    "with w = 2 there is a single stable fixed point"
    fps = fixed_points(2.0, 0.0)
    assert len(fps) == 1 and fps[0][1]

test_three()
test_values()
test_weak()
print("All checks passed")

### Exercise: Predicting the recovery time

At the high stable fixed point for w = 8, compute the slope `slope_hi` and the predicted recovery time constant `tau_pred` = tau / |slope|. Then start the simulation 0.01 below the fixed point, integrate with Euler steps of 0.01 ms for 100 ms, and measure `tau_measured` from the slope of log |r − r*| over the first 30 ms.

In [ ]:
r_hi = fixed_points(8.0, 0.0)[-1][0]
slope_hi = ...
tau_pred = ...
dt = 0.01
ts = np.arange(0, 100, dt)
r = np.empty(len(ts)); r[0] = r_hi - 0.01
for k in range(1, len(ts)):
    r[k] = r[k - 1] + dt / tau * f(r[k - 1], 8.0, 0.0)
early = ts < 30
tau_measured = ...
print(f"high fixed point {r_hi:.4f}, slope {slope_hi:.3f}: predicted {tau_pred:.2f} ms, measured {tau_measured:.2f} ms")

In [ ]:
import numpy as np
def test_slope():
    "slope_hi is the derivative of f at the high fixed point, to 1e-4"
    r_hi = fixed_points(8.0, 0.0)[-1][0]
    assert abs(slope_hi - (f(r_hi + 1e-6, 8.0, 0.0) - f(r_hi - 1e-6, 8.0, 0.0)) / 2e-6) < 1e-4

def test_match():
    "the time constant measured from the simulation is within 5% of the prediction"
    assert abs(tau_measured - tau_pred) / tau_pred < 0.05

test_slope()
test_match()
print("All checks passed")

### Exercise: Switching the population on

Write `run(amp)`, which simulates 300 ms from r = 0 with w = 8, adding an input of size amp from 50 to 70 ms. Store the results for amp = 3 in `r_strong` and amp = 1 in `r_weak`.

In [ ]:
ts = np.arange(0, 300, dt)
def run(amp):
    r = np.zeros(len(ts))
    for k in range(1, len(ts)):
        I = ...                                       # amp between 50 and 70 ms, 0 otherwise
        r[k] = r[k - 1] + dt / tau * f(r[k - 1], 8.0, I)
    return r
r_strong, r_weak = run(3.0), run(1.0)
print(f"before the pulse {r_strong[int(40 / dt)]:.3f};  at 300 ms: strong pulse {r_strong[-1]:.3f}, weak pulse {r_weak[-1]:.3f}")
plt.plot(ts, r_strong, label="pulse 3"); plt.plot(ts, r_weak, label="pulse 1")
plt.axvspan(50, 70, color="orange", alpha=0.2); plt.xlabel("ms"); plt.ylabel("r"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_off_on():
    "the rate starts low (below 0.1) and ends high (above 0.9) after the strong pulse"
    assert r_strong[int(40 / dt)] < 0.1 and r_strong[-1] > 0.9

def test_weak():
    "a pulse too weak to cross the threshold leaves the rate low at the end"
    assert r_weak[-1] < 0.1

test_off_on()
test_weak()
print("All checks passed")